# 🧬 Dark Manifold V19.1 — Dynamic Perturbation Vectors

## V19 vs V19.1

| Aspect | V19 (Static) | V19.1 (Dynamic) |
|--------|--------------|------------------|
| Perturbation | Fixed vector for entire trajectory | **Changes over time** |
| Realism | Low — stress doesn't stay constant | **High — matches real biology** |
| Information | Model sees one snapshot | **Model sees evolving environment** |

## Real Biology: Perturbations Are Dynamic

```
┌─────────────────────────────────────────────────────────────────────┐
│                    DYNAMIC PERTURBATIONS                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Heat Shock:                                                       │
│   temp(t) = +8°C × exp(-t/τ)     # Returns to normal               │
│   ────────────────────────────                                     │
│        ╭──────╮                                                    │
│   +8°C │      ╲                                                    │
│        │       ╲____                                               │
│    0°C │────────────────► time                                     │
│                                                                     │
│   Oxidative Stress:                                                 │
│   H2O2(t) = H2O2_0 × exp(-k×t)   # Degraded by catalase            │
│                                                                     │
│   Starvation:                                                       │
│   glucose(t) = glucose_0 × exp(-consumption×t)  # Depletes         │
│                                                                     │
│   Diauxic Shift:                                                    │
│   glucose(t) = step_down at t=30min                                │
│   lactose(t) = step_up at t=30min                                  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## Key Insight

The model now receives **P(t)** at each timestep, not just **P**.

This means it can learn:
- "When temperature is dropping, do X"
- "When glucose is nearly depleted, do Y"
- "Rate of change matters, not just absolute value"

In [ ]:
#@title 1️⃣ Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time
import json
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"🔥 PyTorch: {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#@title 2️⃣ Configuration

# Model architecture
HIDDEN_DIM = 256
N_REACTIONS = 100
PERTURBATION_DIM = 6  # [temp, d_temp/dt, oxidative, carbon, growth, time]
MODULATION_DIM = 64

# Training
N_EPOCHS = 2000
LEARNING_RATE = 1e-3
DT = 0.02

# Time settings
N_TIMEPOINTS = 20
MAX_TIME = 3.0  # hours

print("📋 V19.1 Configuration:")
print(f"   Perturbation dim: {PERTURBATION_DIM} (includes derivatives!)")
print(f"   Timepoints: {N_TIMEPOINTS} over {MAX_TIME} hours")

In [ ]:
#@title 3️⃣ Define Dynamic Perturbation Functions

def create_perturbation_trajectory(condition_type, times):
    """
    Create TIME-VARYING perturbation vectors.
    
    Returns: (n_timepoints, perturbation_dim)
    
    Perturbation vector at each time:
    [temp, d_temp/dt, oxidative, carbon, growth_rate, normalized_time]
    """
    n_t = len(times)
    P = np.zeros((n_t, PERTURBATION_DIM))
    
    # Normalized time (0 to 1)
    P[:, 5] = times / MAX_TIME
    
    if condition_type == 'heat_pulse':
        # Heat shock that decays back to normal
        # temp(t) = 8 * exp(-t/0.5)
        tau = 0.5
        P[:, 0] = 1.0 * np.exp(-times / tau)  # Temperature (normalized)
        P[:, 1] = -1.0 / tau * np.exp(-times / tau)  # d(temp)/dt
        P[:, 2] = 0.0  # No oxidative
        P[:, 3] = 0.8  # Normal carbon
        P[:, 4] = 0.3 + 0.5 * (1 - np.exp(-times / tau))  # Growth recovers
        
    elif condition_type == 'heat_sustained':
        # Sustained heat (like incubator shift)
        P[:, 0] = 1.0 * np.ones(n_t)  # Constant high temp
        P[:, 1] = 0.0  # No change
        P[:, 2] = 0.0
        P[:, 3] = 0.8
        P[:, 4] = 0.4 * np.ones(n_t)  # Reduced growth
        
    elif condition_type == 'heat_mild':
        # INTERPOLATION: Mild heat pulse
        tau = 0.7
        P[:, 0] = 0.5 * np.exp(-times / tau)  # Half the intensity
        P[:, 1] = -0.5 / tau * np.exp(-times / tau)
        P[:, 2] = 0.0
        P[:, 3] = 0.8
        P[:, 4] = 0.5 + 0.4 * (1 - np.exp(-times / tau))
        
    elif condition_type == 'cold_pulse':
        # Cold shock that recovers
        tau = 0.8
        P[:, 0] = -1.0 * np.exp(-times / tau)
        P[:, 1] = 1.0 / tau * np.exp(-times / tau)  # Warming up
        P[:, 2] = 0.0
        P[:, 3] = 0.8
        P[:, 4] = 0.2 + 0.6 * (1 - np.exp(-times / tau))
        
    elif condition_type == 'cold_mild':
        # INTERPOLATION: Mild cold
        tau = 0.6
        P[:, 0] = -0.5 * np.exp(-times / tau)
        P[:, 1] = 0.5 / tau * np.exp(-times / tau)
        P[:, 2] = 0.0
        P[:, 3] = 0.8
        P[:, 4] = 0.4 + 0.5 * (1 - np.exp(-times / tau))
        
    elif condition_type == 'oxidative_pulse':
        # H2O2 bolus that gets degraded
        k_cat = 2.0  # Catalase rate
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 1.0 * np.exp(-k_cat * times)  # H2O2 decays
        P[:, 3] = 0.8
        P[:, 4] = 0.3 + 0.5 * (1 - np.exp(-times))
        
    elif condition_type == 'oxidative_chronic':
        # Sustained low-level oxidative stress
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.3 * np.ones(n_t)  # Constant low ROS
        P[:, 3] = 0.8
        P[:, 4] = 0.6 * np.ones(n_t)
        
    elif condition_type == 'oxidative_mild':
        # INTERPOLATION: Mild oxidative pulse
        k_cat = 1.5
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.5 * np.exp(-k_cat * times)
        P[:, 3] = 0.8
        P[:, 4] = 0.4 + 0.4 * (1 - np.exp(-times))
        
    elif condition_type == 'starvation_gradual':
        # Glucose gradually depletes
        consumption_rate = 1.0
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.0
        P[:, 3] = 0.9 * np.exp(-consumption_rate * times)  # Glucose depletes
        P[:, 4] = 0.8 * np.exp(-consumption_rate * times * 0.8)  # Growth slows
        
    elif condition_type == 'starvation_sudden':
        # Sudden glucose removal (wash)
        t_starve = 0.3
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.0
        P[:, 3] = np.where(times < t_starve, 0.9, 0.05)  # Step down
        P[:, 4] = np.where(times < t_starve, 0.8, 0.1 + 0.2 * (times - t_starve))
        
    elif condition_type == 'starvation_partial':
        # INTERPOLATION: Partial starvation
        consumption_rate = 0.5
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.0
        P[:, 3] = 0.9 * np.exp(-consumption_rate * times) + 0.2  # Doesn't go to zero
        P[:, 4] = 0.6 * np.exp(-consumption_rate * times * 0.5) + 0.2
        
    elif condition_type == 'diauxic_shift':
        # Glucose runs out, switch to lactose
        t_shift = 0.5
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.0
        # Glucose depletes, then lactose kicks in
        glucose = 0.9 * np.exp(-3 * np.maximum(0, times - t_shift + 0.3))
        lactose = 0.8 * (1 - np.exp(-2 * np.maximum(0, times - t_shift)))
        P[:, 3] = glucose + 0.7 * lactose  # Total carbon
        # Growth dips then recovers
        P[:, 4] = 0.8 - 0.6 * np.exp(-(times - t_shift)**2 / 0.1) + 0.3 * lactose
        
    elif condition_type == 'stationary_entry':
        # Gradual entry into stationary phase
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.1 * (1 - np.exp(-times))  # Slight oxidative stress builds
        P[:, 3] = 0.8 * np.exp(-0.5 * times)  # Carbon depletes
        P[:, 4] = 0.8 * np.exp(-times)  # Growth stops
        
    elif condition_type == 'growth_optimal':
        # Optimal growth (baseline)
        P[:, 0] = 0.0
        P[:, 1] = 0.0
        P[:, 2] = 0.0
        P[:, 3] = 0.9 * np.ones(n_t)  # Plenty of carbon
        P[:, 4] = 0.9 * np.ones(n_t)  # Fast growth
        
    elif condition_type == 'recovery':
        # INTERPOLATION: Recovery from stress
        # Starts stressed, returns to normal
        P[:, 0] = 0.3 * np.exp(-times / 0.3)  # Mild temp that recovers
        P[:, 1] = -0.3 / 0.3 * np.exp(-times / 0.3)
        P[:, 2] = 0.2 * np.exp(-times / 0.5)  # Mild oxidative that clears
        P[:, 3] = 0.5 + 0.4 * (1 - np.exp(-times))  # Carbon restores
        P[:, 4] = 0.3 + 0.6 * (1 - np.exp(-times))  # Growth recovers
        
    else:
        raise ValueError(f"Unknown condition: {condition_type}")
    
    return P

# Define training and test conditions
TRAIN_CONDITIONS = [
    'heat_pulse', 'heat_sustained',
    'cold_pulse',
    'oxidative_pulse', 'oxidative_chronic',
    'starvation_gradual', 'starvation_sudden',
    'diauxic_shift',
    'stationary_entry',
    'growth_optimal'
]

TEST_CONDITIONS = [
    'heat_mild',       # Interpolation: between heat_pulse and growth
    'cold_mild',       # Interpolation: between cold_pulse and growth
    'oxidative_mild',  # Interpolation: between oxidative_pulse and chronic
    'starvation_partial',  # Interpolation: between gradual and optimal
    'recovery'         # Interpolation: combination of multiple stresses
]

print(f"✅ Defined {len(TRAIN_CONDITIONS)} training + {len(TEST_CONDITIONS)} test conditions")

In [ ]:
#@title 4️⃣ Visualize Dynamic Perturbations

times = np.linspace(0, MAX_TIME, N_TIMEPOINTS)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

all_conditions = TRAIN_CONDITIONS + TEST_CONDITIONS

for ax, cond in zip(axes.flat, all_conditions[:12]):
    P = create_perturbation_trajectory(cond, times)
    
    ax.plot(times, P[:, 0], 'r-', label='Temp', lw=2)
    ax.plot(times, P[:, 2], 'b-', label='Oxidative', lw=2)
    ax.plot(times, P[:, 3], 'g-', label='Carbon', lw=2)
    ax.plot(times, P[:, 4], 'k--', label='Growth', lw=2)
    
    is_test = cond in TEST_CONDITIONS
    color = 'red' if is_test else 'black'
    marker = '🧪 TEST' if is_test else '🏋️ TRAIN'
    ax.set_title(f"{marker}\n{cond}", fontsize=10, color=color)
    ax.set_xlabel('Time (h)')
    ax.set_ylim(-1.2, 1.2)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Dynamic Perturbation Vectors — Time-Varying Stress Signals', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v19_1_perturbations.png', dpi=150)
plt.show()
print("💾 Saved: v19_1_perturbations.png")

In [ ]:
#@title 5️⃣ Create Datasets with Dynamic Perturbations

def create_metabolite_response(perturbation_trajectory, times):
    """
    Generate metabolite trajectory that RESPONDS to dynamic perturbation.
    
    The metabolite dynamics depend on the CURRENT perturbation state,
    not just initial conditions.
    """
    metabolites = [
        # Glycolysis (8)
        'G6P', 'F6P', 'FBP', 'DHAP', 'G3P', '3PG', 'PEP', 'PYR',
        # TCA (7)
        'CIT', 'ISOCIT', 'AKG', 'SUC', 'FUM', 'MAL', 'OAA',
        # PPP (4)
        '6PG', 'R5P', 'RU5P', 'E4P',
        # Amino acids (20)
        'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLU', 'GLN', 'GLY', 'HIS', 'ILE',
        'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL',
        # Energy (7)
        'ATP', 'ADP', 'AMP', 'GTP', 'NAD', 'NADH', 'NADPH',
        # Signaling (4)
        'cAMP', 'ppGpp', 'TREH', 'GSH'
    ]
    
    n_mets = len(metabolites)
    n_t = len(times)
    P = perturbation_trajectory  # (n_t, 6)
    
    data = np.ones((n_t, n_mets))
    
    # Extract perturbation components
    temp = P[:, 0]
    d_temp = P[:, 1]
    oxid = P[:, 2]
    carbon = P[:, 3]
    growth = P[:, 4]
    
    for i, met in enumerate(metabolites):
        
        # === GLYCOLYSIS: tracks carbon availability ===
        if met in ['G6P', 'F6P', 'FBP']:
            data[:, i] = 0.2 + 0.8 * carbon  # Direct carbon dependence
            
        elif met == 'PYR':
            data[:, i] = 0.3 + 0.5 * carbon * growth + 0.2 * np.abs(temp)
            
        # === TCA: increases when carbon is limiting (gluconeogenesis) ===
        elif met in ['CIT', 'AKG', 'MAL']:
            data[:, i] = 1.0 + 0.8 * (1 - carbon)  # Inverse carbon relationship
            
        # === PPP: responds to oxidative stress ===
        elif met in ['6PG', 'R5P']:
            data[:, i] = 1.0 + 1.5 * oxid  # PPP upregulated for NADPH
            
        # === AMINO ACIDS: accumulate under low growth, spike under stress ===
        elif met in ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLU', 'GLN', 'GLY',
                    'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER',
                    'THR', 'TRP', 'TYR', 'VAL']:
            stress_signal = np.abs(temp) + oxid
            data[:, i] = 1.0 + 0.8 * (1 - growth) + 0.5 * stress_signal
            
        # === ATP: depleted by stress, requires carbon ===
        elif met == 'ATP':
            stress_drain = 0.3 * np.abs(temp) + 0.4 * oxid
            carbon_supply = 0.3 * carbon
            data[:, i] = np.maximum(0.3, 1.0 - stress_drain + carbon_supply)
            
        # === NADPH: consumed by oxidative stress, produced by PPP ===
        elif met == 'NADPH':
            # Initial consumption, then PPP recovery
            consumption = 0.5 * oxid
            production = 0.3 * oxid * P[:, 5]  # Recovery over time
            data[:, i] = np.maximum(0.2, 1.0 - consumption + production)
            
        # === cAMP: spikes when carbon is low ===
        elif met == 'cAMP':
            data[:, i] = 1.0 + 4.0 * (1 - carbon) ** 2  # Nonlinear response
            
        # === ppGpp: stringent response to starvation ===
        elif met == 'ppGpp':
            stress = (1 - carbon) + (1 - growth) + 0.3 * np.abs(temp)
            data[:, i] = 1.0 + 2.5 * stress
            
        # === Trehalose: heat shock protectant ===
        elif met == 'TREH':
            # Accumulates during heat, but takes time
            heat_signal = np.maximum(0, temp)
            # Integrate heat signal over time (cumulative)
            cumulative_heat = np.cumsum(heat_signal) * (times[1] - times[0]) if len(times) > 1 else heat_signal
            data[:, i] = 1.0 + 1.5 * np.tanh(cumulative_heat)
            
        # === Glutathione: oxidative stress buffer ===
        elif met == 'GSH':
            # Consumed by ROS, then regenerated
            data[:, i] = 1.0 - 0.4 * oxid + 0.3 * (1 - oxid) * P[:, 5]
    
    # Add small noise
    data = data + 0.03 * np.random.randn(*data.shape)
    data = np.clip(data, 0.1, 8.0)
    
    aa_start = metabolites.index('ALA')
    aa_end = metabolites.index('VAL') + 1
    aa_indices = list(range(aa_start, aa_end))
    
    return data, metabolites, aa_indices


# Generate all datasets
times = np.linspace(0, MAX_TIME, N_TIMEPOINTS)
all_datasets = {}

for cond in TRAIN_CONDITIONS + TEST_CONDITIONS:
    P = create_perturbation_trajectory(cond, times)
    data, metabolites, aa_indices = create_metabolite_response(P, times)
    
    all_datasets[cond] = {
        'data': data,
        'perturbation': P,
        'times': times,
        'metabolites': metabolites,
        'aa_indices': aa_indices,
        'is_train': cond in TRAIN_CONDITIONS,
        'is_test': cond in TEST_CONDITIONS
    }

metabolite_names = metabolites
n_mets = len(metabolite_names)

print(f"✅ Created {len(all_datasets)} datasets")
print(f"   {n_mets} metabolites, {len(aa_indices)} amino acids")
print(f"   {N_TIMEPOINTS} timepoints, perturbation dim = {PERTURBATION_DIM}")

In [ ]:
#@title 6️⃣ V19.1 Model — Dynamic Perturbation Aware

class EnvironmentEncoder(nn.Module):
    """Encodes time-varying perturbation into modulation signal"""
    
    def __init__(self, perturbation_dim, modulation_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(perturbation_dim, modulation_dim),
            nn.LayerNorm(modulation_dim),
            nn.GELU(),
            nn.Linear(modulation_dim, modulation_dim),
            nn.GELU(),
            nn.Linear(modulation_dim, modulation_dim)
        )
    
    def forward(self, P):
        return self.net(P)


class DarkManifoldV19_1(nn.Module):
    """
    V19.1: Dynamic perturbation-aware model.
    
    Key difference from V19: perturbation changes at EACH timestep.
    """
    
    def __init__(self, n_mets, n_reactions, hidden_dim, perturbation_dim, modulation_dim):
        super().__init__()
        self.n_mets = n_mets
        self.n_reactions = n_reactions
        
        # Environment encoder
        self.env_encoder = EnvironmentEncoder(perturbation_dim, modulation_dim)
        
        # Stoichiometry
        self.S = nn.Parameter(torch.randn(n_mets, n_reactions) * 0.1)
        
        # Flux network: (metabolites + environment) → fluxes
        self.flux_net = nn.Sequential(
            nn.Linear(n_mets + modulation_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions)
        )
        
        # Environment-modulated degradation
        self.deg_base = nn.Parameter(torch.zeros(n_mets) - 1.0)
        self.deg_modulator = nn.Linear(modulation_dim, n_mets)
        
        # Regulation network
        self.reg_net = nn.Sequential(
            nn.Linear(n_mets + modulation_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_mets),
            nn.Tanh()
        )
        
        # Environment-dependent homeostasis
        self.baseline_base = nn.Parameter(torch.ones(n_mets))
        self.baseline_mod = nn.Linear(modulation_dim, n_mets)
        self.homeo_str = nn.Parameter(torch.tensor(0.1))
    
    def forward(self, M, P_t, dt):
        """
        Single step with current perturbation.
        
        Args:
            M: Metabolites (batch, n_mets)
            P_t: Perturbation at current time (batch, perturbation_dim)
            dt: Time step
        """
        # Encode current environment
        env = self.env_encoder(P_t)
        
        # Flux prediction
        flux_input = torch.cat([M, env], dim=-1)
        v = self.flux_net(flux_input)
        stoich = torch.matmul(v, self.S.T)
        
        # Modulated degradation
        deg_rates = torch.exp(self.deg_base + 0.3 * self.deg_modulator(env)).clamp(0.01, 0.5)
        deg = deg_rates * M
        
        # Environment-aware regulation
        reg_input = torch.cat([M, env], dim=-1)
        reg = self.reg_net(reg_input) * M * 0.3
        
        # Dynamic homeostasis setpoint
        baseline = self.baseline_base + 0.5 * torch.tanh(self.baseline_mod(env))
        homeo = self.homeo_str.clamp(0.01, 0.3) * (baseline - M)
        
        # Integration
        dM = stoich - deg + reg + homeo
        M_new = torch.clamp(M + dt * dM, 0.05, 15.0)
        
        return M_new
    
    def simulate(self, M0, P_trajectory, times, dt=0.01):
        """
        Simulate with TIME-VARYING perturbation.
        
        Args:
            M0: Initial metabolites (batch, n_mets)
            P_trajectory: Perturbation at each timepoint (n_times, perturbation_dim)
            times: Timepoints (n_times,)
            dt: Integration step
        """
        batch_size = M0.shape[0]
        device = M0.device
        
        traj = [M0]
        M = M0.clone()
        t = 0.0
        t_idx = 0
        
        for i, target in enumerate(times[1:], 1):
            # Get perturbation for this segment (interpolate if needed)
            P_t = P_trajectory[i:i+1, :].expand(batch_size, -1)
            
            while t < target - 1e-6:
                step = min(dt, target - t)
                M = self.forward(M, P_t, step)
                t += step
            
            traj.append(M.clone())
        
        return torch.stack(traj, dim=1)


# Initialize
model = DarkManifoldV19_1(
    n_mets=n_mets,
    n_reactions=N_REACTIONS,
    hidden_dim=HIDDEN_DIM,
    perturbation_dim=PERTURBATION_DIM,
    modulation_dim=MODULATION_DIM
).to(device)

print(f"✅ V19.1 Model: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
#@title 7️⃣ Training

def train_v19_1(model, train_datasets, n_epochs, lr, dt, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)
    
    model.train()
    losses = []
    
    for epoch in tqdm(range(n_epochs), desc="Training"):
        epoch_loss = 0.0
        
        for cond_name, ds in train_datasets.items():
            data = torch.tensor(ds['data'], dtype=torch.float32, device=device)
            P_traj = torch.tensor(ds['perturbation'], dtype=torch.float32, device=device)
            times = torch.tensor(ds['times'], dtype=torch.float32, device=device)
            
            optimizer.zero_grad()
            
            M0 = data[0:1, :]
            pred = model.simulate(M0, P_traj, times, dt=dt).squeeze(0)
            
            # Multi-scale loss
            loss_mse = F.mse_loss(pred, data)
            loss_log = F.mse_loss(torch.log(pred + 0.01), torch.log(data + 0.01))
            loss_deriv = F.mse_loss(pred[1:] - pred[:-1], data[1:] - data[:-1])
            
            loss = loss_mse + 0.5 * loss_log + 0.3 * loss_deriv
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        scheduler.step()
        losses.append(epoch_loss / len(train_datasets))
    
    return losses


def evaluate_v19_1(model, dataset, dt, device):
    model.eval()
    with torch.no_grad():
        data = torch.tensor(dataset['data'], dtype=torch.float32, device=device)
        P_traj = torch.tensor(dataset['perturbation'], dtype=torch.float32, device=device)
        times = torch.tensor(dataset['times'], dtype=torch.float32, device=device)
        
        M0 = data[0:1, :]
        pred = model.simulate(M0, P_traj, times, dt=dt).squeeze(0)
        
        pred_np = pred.cpu().numpy()
        true_np = data.cpu().numpy()
        
        overall = np.corrcoef(pred_np.flatten(), true_np.flatten())[0, 1]
        if np.isnan(overall): overall = 0.0
        
        met_corrs = {}
        for i, name in enumerate(dataset['metabolites']):
            if np.std(true_np[:, i]) > 0.01 and np.std(pred_np[:, i]) > 0.01:
                c = np.corrcoef(pred_np[:, i], true_np[:, i])[0, 1]
                met_corrs[name] = c if not np.isnan(c) else 0.0
            else:
                met_corrs[name] = 0.0
        
        aa_corrs = [met_corrs[dataset['metabolites'][i]] for i in dataset['aa_indices']]
        aa_avg = np.mean(aa_corrs)
        
        return {'overall': overall, 'aa_avg': aa_avg, 'met_corrs': met_corrs, 
                'pred': pred_np, 'true': true_np}

print("✅ Training functions defined")

In [ ]:
#@title 8️⃣ Train! 🚀

print("="*70)
print("TRAINING V19.1 — Dynamic Perturbation Model")
print("="*70)

train_datasets = {k: v for k, v in all_datasets.items() if v['is_train']}

print(f"\n🏋️ Training on {len(train_datasets)} conditions")
print(f"🧪 Will test on {len(TEST_CONDITIONS)} interpolation conditions")

start = time.time()
losses = train_v19_1(model, train_datasets, N_EPOCHS, LEARNING_RATE, DT, device)
train_time = time.time() - start

print(f"\n⏱️ Training: {train_time/60:.1f} min")
print(f"📉 Final loss: {losses[-1]:.4f}")

In [ ]:
#@title 9️⃣ Evaluate All Conditions

print("\n" + "="*70)
print("EVALUATION — Train vs Interpolation Test")
print("="*70)

all_results = {}
train_corrs, test_corrs = [], []

print(f"\n{'Condition':<25} {'Type':<10} {'Overall':>10} {'AA Avg':>10}")
print("-" * 60)

for name, ds in all_datasets.items():
    result = evaluate_v19_1(model, ds, DT, device)
    all_results[name] = result
    
    if ds['is_train']:
        ctype = "🏋️ TRAIN"
        train_corrs.append(result['overall'])
    else:
        ctype = "🧪 TEST"
        test_corrs.append(result['overall'])
    
    status = "✅" if result['overall'] > 0.7 else "⚠️" if result['overall'] > 0.4 else "❌"
    print(f"{status} {name:<22} {ctype:<10} {result['overall']:>10.4f} {result['aa_avg']:>10.4f}")

print("-" * 60)
print(f"{'TRAIN MEAN':<35} {np.mean(train_corrs):>10.4f}")
print(f"{'TEST MEAN (interpolation)':<35} {np.mean(test_corrs):>10.4f}")
print(f"{'GENERALIZATION GAP':<35} {np.mean(train_corrs) - np.mean(test_corrs):>10.4f}")

In [ ]:
#@title 🔟 Final Comparison: All Versions

v19_1_train = np.mean(train_corrs)
v19_1_test = np.mean(test_corrs)
v19_1_gap = v19_1_train - v19_1_test

print(f"""
╔════════════════════════════════════════════════════════════════════════╗
║                    COMPLETE VERSION COMPARISON                          ║
╠════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║   Version │ Train  │ Test   │  Gap   │ Test Type                       ║
║   ─────────────────────────────────────────────────────────────────    ║
║   V16     │  ~0.1  │  0.033 │ ~0.07  │ Extrapolation (diff stress)     ║
║   V17     │  0.85  │  0.020 │  0.83  │ Extrapolation (diff stress)     ║
║   V18     │  N/A   │ -0.001 │  N/A   │ Zero-shot (random weights)      ║
║   V19     │   ?    │   ?    │   ?    │ Interpolation (static perturb)  ║
║   V19.1   │ {v19_1_train:.3f}  │ {v19_1_test:.3f}  │ {v19_1_gap:.3f}  │ Interpolation (DYNAMIC perturb) ║
║                                                                        ║
╚════════════════════════════════════════════════════════════════════════╝
""")

# Interpretation
if v19_1_test > 0.7:
    print("🏆 BREAKTHROUGH! Dynamic perturbations enable strong generalization!")
elif v19_1_test > 0.5:
    print("✅ SUCCESS! Meaningful interpolation with dynamic perturbations.")
elif v19_1_test > 0.3:
    print("⚠️ PROGRESS. Better than V16-V18, but room to improve.")
else:
    print("❌ Still struggling. May need more training data or architecture changes.")

print(f"\n📊 Key: Test correlation improved from 0.02 (V17) to {v19_1_test:.3f} (V19.1)")
print(f"   Gap reduced from 0.83 (V17) to {v19_1_gap:.3f} (V19.1)")

In [ ]:
#@title 1️⃣1️⃣ Visualize Results

fig = plt.figure(figsize=(16, 12))

# 1. Training curve
ax1 = fig.add_subplot(2, 2, 1)
ax1.semilogy(losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('V19.1 Training Loss', fontsize=12)
ax1.grid(True, alpha=0.3)

# 2. Bar chart
ax2 = fig.add_subplot(2, 2, 2)
names = list(all_results.keys())
corrs = [all_results[n]['overall'] for n in names]
colors = ['blue' if all_datasets[n]['is_train'] else 'red' for n in names]
ax2.bar(range(len(names)), corrs, color=colors, edgecolor='black', alpha=0.7)
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=7, rotation=45)
ax2.axhline(y=np.mean(train_corrs), color='blue', linestyle='--', label=f'Train: {np.mean(train_corrs):.2f}')
ax2.axhline(y=np.mean(test_corrs), color='red', linestyle='--', label=f'Test: {np.mean(test_corrs):.2f}')
ax2.set_ylabel('Correlation')
ax2.set_title('All Conditions (Blue=Train, Red=Test)', fontsize=12)
ax2.legend()
ax2.set_ylim(-0.2, 1.0)
ax2.grid(axis='y', alpha=0.3)

# 3. Example test trajectory
ax3 = fig.add_subplot(2, 2, 3)
test_cond = 'heat_mild'
result = all_results[test_cond]
times = all_datasets[test_cond]['times']
for met in ['G6P', 'ATP', 'GLU', 'TREH', 'ppGpp']:
    idx = metabolite_names.index(met)
    ax3.plot(times, result['true'][:, idx], 'o-', label=f'{met} (true)', lw=2, ms=4)
    ax3.plot(times, result['pred'][:, idx], 's--', label=f'{met} (pred)', lw=1.5, ms=3, alpha=0.7)
ax3.set_xlabel('Time (hours)')
ax3.set_ylabel('Concentration')
ax3.set_title(f'TEST: {test_cond} (r={result["overall"]:.3f})', fontsize=12)
ax3.legend(fontsize=7, ncol=2)
ax3.grid(True, alpha=0.3)

# 4. Dynamic perturbation for test condition
ax4 = fig.add_subplot(2, 2, 4)
P = all_datasets[test_cond]['perturbation']
ax4.plot(times, P[:, 0], 'r-', label='Temperature', lw=2)
ax4.plot(times, P[:, 2], 'b-', label='Oxidative', lw=2)
ax4.plot(times, P[:, 3], 'g-', label='Carbon', lw=2)
ax4.plot(times, P[:, 4], 'k--', label='Growth', lw=2)
ax4.set_xlabel('Time (hours)')
ax4.set_ylabel('Perturbation value')
ax4.set_title(f'Dynamic Perturbation: {test_cond}', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle(f'Dark Manifold V19.1 — Dynamic Perturbations\nTrain: {v19_1_train:.3f}, Test: {v19_1_test:.3f}, Gap: {v19_1_gap:.3f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v19_1_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Saved: v19_1_results.png")

In [ ]:
#@title 1️⃣2️⃣ Test Conditions Detail

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, test_cond in zip(axes.flat, TEST_CONDITIONS + ['heat_pulse']):
    if test_cond not in all_results:
        continue
    result = all_results[test_cond]
    times = all_datasets[test_cond]['times']
    
    for met in ['G6P', 'ATP', 'GLU', 'TREH']:
        idx = metabolite_names.index(met)
        ax.plot(times, result['true'][:, idx], 'o-', label=f'{met} (true)', lw=2, ms=4)
        ax.plot(times, result['pred'][:, idx], 's--', label=f'{met} (pred)', lw=1.5, ms=3, alpha=0.7)
    
    is_test = all_datasets[test_cond]['is_test']
    status = "✅" if result['overall'] > 0.7 else "⚠️" if result['overall'] > 0.4 else "❌"
    marker = "🧪 TEST" if is_test else "🏋️ TRAIN"
    ax.set_title(f"{status} {marker}: {test_cond}\nr={result['overall']:.3f}", fontsize=10)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Concentration')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.suptitle('V19.1 — Test Conditions (Interpolations) vs Training Example', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v19_1_test_detail.png', dpi=150)
plt.show()
print("\n💾 Saved: v19_1_test_detail.png")

In [ ]:
#@title 1️⃣3️⃣ Save Results

save_dict = {
    'version': 'V19.1',
    'method': 'Dynamic Perturbation-Aware Model',
    'key_improvement': 'Time-varying perturbation vectors (not static)',
    'perturbation_dim': PERTURBATION_DIM,
    'perturbation_components': ['temp', 'd_temp/dt', 'oxidative', 'carbon', 'growth', 'time'],
    'train_conditions': TRAIN_CONDITIONS,
    'test_conditions': TEST_CONDITIONS,
    'results': {
        'train_mean': float(v19_1_train),
        'test_mean': float(v19_1_test),
        'gap': float(v19_1_gap),
        'per_condition': {
            name: {'overall': float(r['overall']), 'aa_avg': float(r['aa_avg'])}
            for name, r in all_results.items()
        }
    },
    'comparison': {
        'v17_test': 0.020,
        'v18_zero_shot': -0.001,
        'v19_1_test': float(v19_1_test),
        'improvement': float(v19_1_test - 0.020)
    }
}

with open('v19_1_results.json', 'w') as f:
    json.dump(save_dict, f, indent=2)
print("💾 Saved: v19_1_results.json")

torch.save({'model_state_dict': model.state_dict(), 'config': save_dict}, 'v19_1_model.pt')
print("💾 Saved: v19_1_model.pt")

from google.colab import files
files.download('v19_1_results.json')
files.download('v19_1_results.png')
files.download('v19_1_test_detail.png')
files.download('v19_1_perturbations.png')
files.download('v19_1_model.pt')

# 📌 V19.1 Summary

## Key Innovation: Dynamic Perturbations

| V19 (Static) | V19.1 (Dynamic) |
|--------------|------------------|
| P = [0.8, 0, 0.3, ...] constant | P(t) changes over time |
| Model sees snapshot | Model sees trajectory |
| "Heat stress is ON" | "Heat is decaying, temp dropping" |

## Perturbation Vector Now Includes:

1. **Temperature** — current value
2. **d(Temperature)/dt** — rate of change (warming/cooling)
3. **Oxidative load** — H2O2 level
4. **Carbon availability** — glucose level  
5. **Growth rate** — current μ
6. **Normalized time** — where in experiment

## Why This Is More Realistic

Real cells respond to:
- Not just "is it hot" but "is it getting hotter or cooling down"
- Not just "is there ROS" but "is the oxidative stress increasing or being cleared"
- The TRAJECTORY of environmental change, not just the current state

## Next Steps

If V19.1 works:
1. Test on real experimental data with measured environmental parameters
2. Add more perturbation dimensions (pH, osmolarity, antibiotics)
3. Try extrapolation (slightly beyond training range)